## Import Libraries

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os
import sys

## Configure Pyspark Enviroment in this Project

In [2]:
# Configure Spark
hadoop_home = "C:/hadoop"
os.environ['HADOOP_HOME'] = hadoop_home
os.environ['PATH'] = os.path.join(hadoop_home, 'bin') + os.pathsep + os.environ['PATH']
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

## Create SparkSession

In [3]:
# create spark session
spark = SparkSession.builder.appName("XTD_Labs_Historical_Data_Processing").master("local[*]").config("spark.driver.memory", "4g").getOrCreate()
# Ignore Warnings
spark.sparkContext.setLogLevel("ERROR")

spark

## Read all JSON files from the Bronze Layer

In [4]:
# Define the path for the bronze layer (if not created)
BRONZE_DIR = "data/bronze"

# Read all the raw json file from the bronze folder
raw_df = spark.read.json(f'./{BRONZE_DIR}/*json', multiLine=True)
# Count the number of data point per 30-minutes interval
data_count = raw_df.count()
print(f"Total Data Point for 30mins interval found: {data_count}")

Total Data Point for 30mins interval found: 50360


## SILVER LAYER I: FLATTENING THE DATA POINTS

In [5]:
# SILVER LAYER I: FLATTENING THE DATA POINTS
exploded_df = raw_df.select(
    F.col("from").alias("timestamp"),
    F.explode(F.col("regions")).alias("region")
)

exploded_df.show(5)

+-----------------+--------------------+
|        timestamp|              region|
+-----------------+--------------------+
|2023-09-21T23:30Z|{Scottish Hydro E...|
|2023-09-21T23:30Z|{SP Distribution,...|
|2023-09-21T23:30Z|{Electricity Nort...|
|2023-09-21T23:30Z|{NPG North East, ...|
|2023-09-21T23:30Z|{NPG Yorkshire, [...|
+-----------------+--------------------+
only showing top 5 rows


In [6]:
# Unpack
silver_df = exploded_df.select(
    "timestamp",
    F.col("region.regionid").alias('regionid'),
    F.col("region.shortname").alias('shortname'),
    F.col("region.dnoregion").alias('dno'),
    F.col("region.intensity.forecast").alias("intensity"),
    F.col("region.intensity.index").alias("index"),
    F.explode(F.col("region.generationmix")).alias("mix")
)

silver_df.show(5)

+-----------------+--------+--------------+--------------------+---------+--------+--------------+
|        timestamp|regionid|     shortname|                 dno|intensity|   index|           mix|
+-----------------+--------+--------------+--------------------+---------+--------+--------------+
|2023-09-21T23:30Z|       1|North Scotland|Scottish Hydro El...|        0|very low|{biomass, 0.0}|
|2023-09-21T23:30Z|       1|North Scotland|Scottish Hydro El...|        0|very low|   {coal, 0.0}|
|2023-09-21T23:30Z|       1|North Scotland|Scottish Hydro El...|        0|very low|{imports, 0.0}|
|2023-09-21T23:30Z|       1|North Scotland|Scottish Hydro El...|        0|very low|    {gas, 0.0}|
|2023-09-21T23:30Z|       1|North Scotland|Scottish Hydro El...|        0|very low|{nuclear, 0.0}|
+-----------------+--------+--------------+--------------------+---------+--------+--------------+
only showing top 5 rows


## SILVER LAYER II: PIVOT

In [7]:
# SILVER LAYER II: pIVOT
silver_df_pivoted = silver_df.groupBy("regionid", "shortname", "dno", "timestamp", "intensity", "index").pivot("mix.fuel").agg(F.first("mix.perc"))

silver_df_pivoted.show(5)

+--------+--------------------+---------------+-----------------+---------+--------+-------+----+----+-----+-------+-------+-----+-----+----+
|regionid|           shortname|            dno|        timestamp|intensity|   index|biomass|coal| gas|hydro|imports|nuclear|other|solar|wind|
+--------+--------------------+---------------+-----------------+---------+--------+-------+----+----+-----+-------+-------+-----+-----+----+
|       2|      South Scotland|SP Distribution|2023-09-26T19:00Z|       12|very low|    2.1| 0.0| 2.5|  7.2|    0.0|   26.3|  0.0|  0.0|61.9|
|       4|  North East England| NPG North East|2023-09-16T11:30Z|        4|very low|    0.4| 0.0| 0.5|  0.1|   59.4|   25.4|  0.0|  2.0|12.2|
|       6|North Wales & Mer...|      SP Manweb|2024-03-02T21:30Z|       54|     low|    3.1| 0.0|12.6|  3.3|    4.4|   23.1|  0.0|  0.0|53.5|
|       4|  North East England| NPG North East|2023-01-02T19:30Z|        2|very low|    0.1| 0.0| 0.0|  0.1|   51.7|   46.4|  0.0|  0.0| 1.6|
|     

## 4. GOLD LAYER: Aggregate to Daily Averages

In [8]:
# 4. GOLD LAYER: Aggregate to Daily Averages
print('Starting Gold Layer Aggregation')
gold_df = silver_df_pivoted.withColumn("date_recorded", F.to_date("timestamp")) \
.groupBy("regionid", "date_recorded") \
    .agg(
        F.first("shortname").alias("shortname"),
        F.first("dno").alias("dno"),
        F.round(F.mean("intensity"), 2).alias("intensity_avg"),
        F.mode("index").alias("index_mode"),
        F.round(F.mean("biomass"), 2).alias("fuel_biomass"),
        F.round(F.mean("coal"), 2).alias("fuel_coal"),
        F.round(F.mean("gas"), 2).alias("fuel_gas"),
        F.round(F.mean("hydro"), 2).alias("fuel_hydro"),
        F.round(F.mean("imports"), 2).alias("fuel_imports"),
        F.round(F.mean("nuclear"), 2).alias("fuel_nuclear"),
        F.round(F.mean("other"), 2).alias("fuel_other"),
        F.round(F.mean("solar"), 2).alias("fuel_solar"),
        F.round(F.mean("wind"), 2).alias("fuel_wind")
    ).orderBy("date_recorded", "regionid")

final_count = gold_df.count()
print(f"Total Data Point processed and saved: {final_count}")

Starting Gold Layer Aggregation
Total Data Point processed and saved: 18648


In [9]:
gold_df.show(5)

+--------+-------------+------------------+--------------------+-------------+----------+------------+---------+--------+----------+------------+------------+----------+----------+---------+
|regionid|date_recorded|         shortname|                 dno|intensity_avg|index_mode|fuel_biomass|fuel_coal|fuel_gas|fuel_hydro|fuel_imports|fuel_nuclear|fuel_other|fuel_solar|fuel_wind|
+--------+-------------+------------------+--------------------+-------------+----------+------------+---------+--------+----------+------------+------------+----------+----------+---------+
|       1|   2021-12-30|    North Scotland|Scottish Hydro El...|          0.0|  very low|         0.0|      0.0|     0.0|      12.6|         0.0|         0.0|       0.0|       0.0|     87.4|
|       2|   2021-12-30|    South Scotland|     SP Distribution|         13.0|  very low|         1.2|      0.0|     3.0|       2.1|         0.0|        63.1|       0.0|       0.0|     30.7|
|       3|   2021-12-30|North West England|El

In [10]:
# Save
gold_df.write.csv("data/gold_carbon_historical.csv", header=True, mode="overwrite")

# see what we processed
gold_df.show(10)


+--------+-------------+--------------------+--------------------+-------------+----------+------------+---------+--------+----------+------------+------------+----------+----------+---------+
|regionid|date_recorded|           shortname|                 dno|intensity_avg|index_mode|fuel_biomass|fuel_coal|fuel_gas|fuel_hydro|fuel_imports|fuel_nuclear|fuel_other|fuel_solar|fuel_wind|
+--------+-------------+--------------------+--------------------+-------------+----------+------------+---------+--------+----------+------------+------------+----------+----------+---------+
|       1|   2021-12-30|      North Scotland|Scottish Hydro El...|          0.0|  very low|         0.0|      0.0|     0.0|      12.6|         0.0|         0.0|       0.0|       0.0|     87.4|
|       2|   2021-12-30|      South Scotland|     SP Distribution|         13.0|  very low|         1.2|      0.0|     3.0|       2.1|         0.0|        63.1|       0.0|       0.0|     30.7|
|       3|   2021-12-30|  North Wes